# PW_Bioacoustics Demo

End-to-end walkthrough of the PW_Bioacoustics pipeline using real bird recordings from the
[Humboldt Aves dataset](https://zenodo.org/records/18563039) (project PPA4, Putumayo, Colombia — CC BY 4.0).

**Pipeline**
```
WAV files + Raven annotations (.txt)
        ↓
PPA4Reader  →  COCO-like annotations JSON
        ↓
build_windows()  →  sliding/balanced windows
        ↓
compute_mel_spectrograms_gpu()  →  .npy spectrogram files
        ↓
ResNetClassifier (PyTorch Lightning)  →  trained checkpoint
        ↓
run_inference_batch()  →  predictions CSV
```

**Two classification modes**
- **Binary** — bird vocalization (`AVEVOC`) vs. background noise  
- **Multiclass** — top-4 species (`CACCEL`, `CYAVIO`, `PSAANG`, `RAMCAR`) vs. noise

**Run this notebook from `PW_Bioacoustics/demo/`**
```bash
cd PW_Bioacoustics/demo
jupyter notebook bioacoustics_demo.ipynb
```

## 0. Setup

In [4]:
import os
import sys
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import soundfile as sf
import torchaudio
import torch

warnings.filterwarnings("ignore")

# ── Verify working directory ────────────────────────────────────────────────
DEMO_DIR = Path(os.getcwd())
if DEMO_DIR.name != "demo":
    raise RuntimeError(
        f"Run this notebook from PW_Bioacoustics/demo/. Currently in: {DEMO_DIR}\n"
        "  cd PW_Bioacoustics/demo && jupyter notebook bioacoustics_demo.ipynb"
    )

PW_BIO_DIR     = DEMO_DIR.parent        # PW_Bioacoustics/
CAMERATRAP_DIR = PW_BIO_DIR.parent      # CameraTraps/

for p in [str(PW_BIO_DIR), str(CAMERATRAP_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Directory layout ────────────────────────────────────────────────────────
DATA_DIR       = DEMO_DIR / "data"
AUDIOS_DIR     = DATA_DIR / "audios"
LABELS_DIR     = DATA_DIR / "labels"
OUTPUT_DIR     = DEMO_DIR / "output"
SPEC_DIR       = OUTPUT_DIR / "spectrograms"   # shared by both modes
BINARY_DIR     = OUTPUT_DIR / "binary"
MULTICLASS_DIR = OUTPUT_DIR / "multiclass"

for d in [AUDIOS_DIR, LABELS_DIR, SPEC_DIR, BINARY_DIR, MULTICLASS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment ready")
print(f"  PW_Bioacoustics : {PW_BIO_DIR}")
print(f"  Data            : {DATA_DIR}")
print(f"  Output          : {OUTPUT_DIR}")

Environment ready
  PW_Bioacoustics : /home/v-druizlopez/CameraTraps/PW_Bioacoustics
  Data            : /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/data
  Output          : /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output


## 1. Data Exploration

Five recordings from project PPA4 (Putumayo, Colombia) annotated with Raven Pro.
Each `.Table.1.selections.txt` file lists annotated sound events with `Begin Time (s)`,
`End Time (s)`, and species `Determination`.

In [5]:
PPA4_FILES = [
    "G021_timelapse_20250623",
    "G021_timelapse_20250622",
    "G040_timelapse_20250625",
    "G040_timelapse_20250629",
    "G010_timelapse_20250629",
]

rows = []
for stem in PPA4_FILES:
    txt_path = LABELS_DIR / f"{stem}.Table.1.selections.txt"
    wav_path = AUDIOS_DIR / f"{stem}.wav"
    df = pd.read_csv(txt_path, delimiter="\t")
    info = sf.info(str(wav_path))
    with_species = df["Determination"].apply(
        lambda x: isinstance(x, str) and x.strip() != ""
    ).sum()
    rows.append({
        "file": stem,
        "duration (min)": round(info.duration / 60, 1),
        "sample_rate (kHz)": info.samplerate // 1000,
        "total_annotations": len(df),
        "with_species_id": int(with_species),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\nTotal annotations : {summary['total_annotations'].sum()}")
print(f"Total duration    : {summary['duration (min)'].sum():.0f} min")

                   file  duration (min)  sample_rate (kHz)  total_annotations  with_species_id
G021_timelapse_20250623             8.0                192                125               72
G021_timelapse_20250622             8.0                192                120               84
G040_timelapse_20250625             8.0                192                 97               49
G040_timelapse_20250629             8.0                192                 81               50
G010_timelapse_20250629             8.0                192                 80               42

Total annotations : 503
Total duration    : 40 min


In [6]:
TOP_SPECIES = ["CACCEL", "CYAVIO", "PSAANG", "RAMCAR"]

species_counts = {}
for stem in PPA4_FILES:
    df = pd.read_csv(LABELS_DIR / f"{stem}.Table.1.selections.txt", delimiter="\t")
    for det in df["Determination"].dropna():
        det = str(det).strip()
        if det:
            species_counts[det] = species_counts.get(det, 0) + 1

top10 = sorted(species_counts.items(), key=lambda x: -x[1])[:10]
labels_plot = [s for s, _ in top10]
counts_plot = [c for _, c in top10]
colors = ["#1976D2" if s in TOP_SPECIES else "#B0BEC5" for s in labels_plot]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels_plot[::-1], counts_plot[::-1], color=colors[::-1])
ax.set_xlabel("Annotation count (across 5 recordings)")
ax.set_title("Top 10 species — blue bars used for multiclass demo")
patch_blue = mpatches.Patch(color="#1976D2", label="Top-4 multiclass species")
patch_grey = mpatches.Patch(color="#B0BEC5", label="Other / background")
ax.legend(handles=[patch_blue, patch_grey], fontsize=8)
plt.tight_layout()
plt.show()

print(f"Top-4 species counts: { {s: species_counts.get(s,0) for s in TOP_SPECIES} }")
total_pos = sum(species_counts.values())
print(f"All annotated events (binary positives): {total_pos}")

Top-4 species counts: {'CACCEL': 22, 'CYAVIO': 38, 'PSAANG': 43, 'RAMCAR': 63}
All annotated events (binary positives): 297


In [8]:
# Visualise one RAMCAR vocalization from G021_timelapse_20250623
WAV_FILE = str(AUDIOS_DIR / "G021_timelapse_20250623.wav")
TXT_FILE = LABELS_DIR / "G021_timelapse_20250623.Table.1.selections.txt"

df_labels = pd.read_csv(TXT_FILE, delimiter="\t")
row = df_labels[df_labels["Determination"] == "RAMCAR"].iloc[0]
t_call_start = float(row["Begin Time (s)"])
t_call_end   = float(row["End Time (s)"])
t_win_start  = max(0.0, (t_call_start + t_call_end) / 2 - 2.5)  # 5s window

info = torchaudio.info(WAV_FILE)
orig_sr = info.sample_rate
SR_TARGET = 48000

waveform, _ = torchaudio.load(
    WAV_FILE,
    frame_offset=int(t_win_start * orig_sr),
    num_frames=int(5.0 * orig_sr),
)
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
if orig_sr != SR_TARGET:
    waveform = torchaudio.functional.resample(waveform, orig_sr, SR_TARGET)

mel_tf = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR_TARGET, n_fft=2048, hop_length=512, n_mels=128,
    f_min=0.0, f_max=SR_TARGET / 2, power=2.0, center=False,
    norm="slaney", mel_scale="slaney",
)
to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80.0)
S_db = to_db(mel_tf(waveform)).squeeze(0).numpy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6),
                                gridspec_kw={"height_ratios": [1, 3]})

t_axis = np.linspace(0, 5.0, waveform.shape[-1])
ax1.plot(t_axis, waveform.squeeze(0).numpy(), lw=0.4, color="#1976D2")
ax1.set_xlim(0, 5.0)
ax1.set_ylabel("Amplitude")
call_rel_start = t_call_start - t_win_start
call_rel_end   = t_call_end   - t_win_start
ax1.axvspan(call_rel_start, call_rel_end, alpha=0.25, color="red")
ax1.set_title("RAMCAR vocalization — G021_timelapse_20250623.wav")

im = ax2.imshow(S_db, origin="lower", aspect="auto",
                extent=[0, 5.0, 0, SR_TARGET / 2], cmap="magma")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Frequency (Hz)")
ax2.axvline(call_rel_start, color="red", lw=1.5, ls="--", label="annotation")
ax2.axvline(call_rel_end,   color="red", lw=1.5, ls="--")
ax2.legend(fontsize=8)
plt.colorbar(im, ax=ax2, label="dB")
plt.tight_layout()
plt.show()

## 2. Build COCO Annotations

`PPA4Reader` follows the same `BaseReader` pattern as `data_reader.py` in the source dataset.
It converts Raven Pro selection tables to the COCO-like JSON that `build_windows()` expects.

- **Binary mode** — every annotated event → `category_id=1` (`AVEVOC`)
- **Multiclass mode** — only the top-4 species → `category_id` 1–4; unlabeled events are skipped
  and will fall into background windows during `build_windows()`

In [9]:
from PytorchWildlife.data.bioacoustics.bioacoustics_annotations import BaseReader, AnnotationCreator


class PPA4Reader(BaseReader):
    """Converts PPA4 Raven Pro annotations to COCO-like JSON.

    Parameters
    ----------
    demo_data_dir : str or Path
        Path to demo/data/  (must contain audios/ and labels/ sub-directories).
    mode : {"binary", "multiclass"}
        - binary     : every AVEVOC event → category_id=1
        - multiclass : top-4 species (CACCEL, CYAVIO, PSAANG, RAMCAR) → category_id 1-4
    """

    TOP_SPECIES = ["CACCEL", "CYAVIO", "PSAANG", "RAMCAR"]  # category_id 1-4

    def __init__(self, demo_data_dir, mode="binary"):
        super().__init__(str(demo_data_dir))
        if mode not in ("binary", "multiclass"):
            raise ValueError("mode must be 'binary' or 'multiclass'")
        self.mode = mode
        self.audio_dir  = Path(demo_data_dir) / "audios"
        self.labels_dir = Path(demo_data_dir) / "labels"
        self.output_path = str(Path(demo_data_dir) / f"{mode}_annotations.json")

    def add_dataset_info(self):
        self.annotation_creator.add_info(
            title="Humboldt Aves — PPA4 demo subset (5 recordings)",
            license="CC BY 4.0",
            description=(
                "5 recordings from project PPA4 (Putumayo, Colombia). "
                "Annotated with Raven Pro. Part of the Humboldt Aves dataset "
                "(https://zenodo.org/records/18563039)."
            ),
        )

    def add_sounds(self):
        # Sort so that sound_id == list-position (required by AnnotationCreator.add_annotation)
        wav_files = sorted(f for f in os.listdir(self.audio_dir) if f.endswith(".wav"))
        for sound_id, filename in enumerate(wav_files):
            file_path = str(self.audio_dir / filename)
            duration, sample_rate = self.annotation_creator._get_duration_and_sample_rate(file_path)
            self.annotation_creator.add_sound(
                id=sound_id,
                file_name_path=file_path,   # absolute path
                duration=duration,
                sample_rate=sample_rate,
                latitude=float("nan"),
                longitude=float("nan"),
            )

    def add_categories(self):
        # Build categories manually to guarantee noise=0
        if self.mode == "binary":
            cats = [
                {"id": 0, "name": "noise",  "supercategory": "background"},
                {"id": 1, "name": "AVEVOC", "supercategory": "BIO"},
            ]
        else:
            cats = [{"id": 0, "name": "noise", "supercategory": "background"}]
            for i, sp in enumerate(self.TOP_SPECIES, start=1):
                cats.append({"id": i, "name": sp, "supercategory": "AVEVOC"})
        self.annotation_creator.data["categories"] = cats

    def add_annotations(self):
        wav_files = sorted(f for f in os.listdir(self.audio_dir) if f.endswith(".wav"))
        anno_id = 0
        for sound_id, wav_file in enumerate(wav_files):
            stem    = os.path.splitext(wav_file)[0]
            txt_path = self.labels_dir / f"{stem}.Table.1.selections.txt"
            if not txt_path.exists():
                continue

            df = pd.read_csv(txt_path, delimiter="\t")
            for _, row in df.iterrows():
                t_min = float(row["Begin Time (s)"])
                t_max = float(row["End Time (s)"])
                determination = str(row.get("Determination", "")).strip()

                if self.mode == "binary":
                    self.annotation_creator.add_annotation(
                        anno_id=anno_id, sound_id=sound_id,
                        category_id=1, category="AVEVOC",
                        supercategory="BIO", t_min=t_min, t_max=t_max,
                    )
                    anno_id += 1
                else:
                    if determination in self.TOP_SPECIES:
                        cat_id = self.TOP_SPECIES.index(determination) + 1
                        self.annotation_creator.add_annotation(
                            anno_id=anno_id, sound_id=sound_id,
                            category_id=cat_id, category=determination,
                            supercategory="AVEVOC", t_min=t_min, t_max=t_max,
                        )
                        anno_id += 1

print("PPA4Reader defined")

PPA4Reader defined


In [10]:
for mode in ("binary", "multiclass"):
    reader = PPA4Reader(DATA_DIR, mode=mode)
    reader.process_dataset()
    print()

Dataset: Humboldt Aves — PPA4 demo subset (5 recordings)
Total species: 2
Total audio recordings: 5
Total annotations: 503
Total duration: 0.67 hours

Dataset: Humboldt Aves — PPA4 demo subset (5 recordings)
Total species: 5
Total audio recordings: 5
Total annotations: 166
Total duration: 0.67 hours



## 3. Binary Classification

**Task:** detect any bird vocalization (`AVEVOC`) vs. background noise.

Pipeline steps:
1. Load config from `binary_config.yaml`
2. `build_windows()` with `strategy="balanced"` and `multiclass=False` → label ∈ {0, 1}
3. `compute_mel_spectrograms_gpu()` → `.npy` files
4. Create train / val / test splits
5. Train `ResNetClassifier` with `num_classes=2`

In [11]:
import yaml
from PytorchWildlife.data.bioacoustics.bioacoustics_configs import load_config

BINARY_CONFIG_PATH = str(DEMO_DIR / "config" / "ppa4_binary.yaml")
os.makedirs(os.path.dirname(BINARY_CONFIG_PATH), exist_ok=True)

binary_cfg_dict = {
    "name": "ppa4_binary",
    "datasets": ["audios"],
    "class_names": {0: "noise", 1: "AVEVOC"},
    "paths": {
        "data_root":        str(DATA_DIR),
        "output_root":      str(BINARY_DIR),
        "spectrograms_dir": str(SPEC_DIR),
        "annotations_file": "binary_annotations.json",
    },
    "audio": {
        "sample_rate":          48000,
        "window_size_sec":      5.0,
        "overlap_sec":          4.0,
        "window_strategy":      "balanced",
        "negative_proportion":  0.5,
        "multiclass":           False,
    },
    "spectrogram": {
        "n_fft":        2048,
        "hop_length":   512,
        "n_mels":       128,
        "top_db":       80.0,
        "fill_highfreq": False,  # downsampling 192→48 kHz, no upsampling gap
        "noise_db_std": 3.0,
        "storage_dtype": "float32",
    },
    "training": {
        "batch_size":     16,
        "num_workers":    2,
        "lr":             1e-4,
        "weight_decay":   1e-4,
        "epochs":         10,
        "backbone":       "resnet18",
        "num_classes":    2,
        "target_size":    [128, 465],
        "normalize":      True,
        "use_specaug":    True,
        "pos_weight":     2.0,
        "conf_threshold": 0.5,
    },
    "splits": {
        "test_size":    0.2,
        "val_size":     0.2,
        "n_splits":     3,
        "random_state": 42,
    },
}

with open(BINARY_CONFIG_PATH, "w") as f:
    yaml.dump(binary_cfg_dict, f, default_flow_style=False, sort_keys=False)

binary_cfg = load_config(BINARY_CONFIG_PATH)
print(f"Config loaded: {binary_cfg.name}")
print(f"  sample_rate    : {binary_cfg.audio.sample_rate} Hz")
print(f"  window_size    : {binary_cfg.audio.window_size_sec}s")
print(f"  strategy       : {binary_cfg.audio.window_strategy}")
print(f"  num_classes    : {binary_cfg.training.num_classes}")
print(f"  backbone       : {binary_cfg.training.backbone}")

Config loaded: ppa4_binary
  sample_rate    : 48000 Hz
  window_size    : 5.0s
  strategy       : balanced
  num_classes    : 2
  backbone       : resnet18


In [12]:
import sys
# Make sure prepare_dataset functions are importable
from prepare_dataset import run_stats, run_windows

run_stats(binary_cfg)
binary_windows = run_windows(binary_cfg)


Step: Dataset Statistics
Loading annotations from: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/data/binary_annotations.json

Dataset Info:
  - title: Humboldt Aves — PPA4 demo subset (5 recordings)
  - license: CC BY 4.0
  - publication_date: None
  - description: 5 recordings from project PPA4 (Putumayo, Colombia). Annotated with Raven Pro. Part of the Humboldt Aves dataset (https://zenodo.org/records/18563039).
  - creators: None
  - version: None
  - url: None

Sounds: 5
  - Total duration: 2400.0s (0.67h)
  - Mean duration: 480.0s
  - Min duration: 480.0s
  - Max duration: 480.0s

Annotations: 503
  - By category: {1: 503}

Categories:
  - 0: noise
  - 1: AVEVOC

Step: Build Windows
Loading existing windows from: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/binary/windows_mapping_4.0overlap.json
Loaded 1006 windows

Label distribution: {1: 503, 0: 503}


In [13]:
# compute_mel_spectrograms_gpu is imported from inference.py (PW_Bioacoustics/)
from inference import compute_mel_spectrograms_gpu

# Translate windows → inference format (add sound_path from annotations JSON)
with open(binary_cfg.paths.annotations_path) as f:
    anno_data = json.load(f)
sounds_map = {s["id"]: s for s in anno_data["sounds"]}

inf_windows = [
    {
        "window_id":  w["window_id"],
        "sound_path": sounds_map[w["sound_id"]]["file_name_path"],
        "start":      w["start"],
        "end":        w["end"],
    }
    for w in binary_windows
    if w["sound_id"] in sounds_map
]

compute_mel_spectrograms_gpu(
    windows=inf_windows,
    sample_rate=binary_cfg.audio.sample_rate,
    n_fft=binary_cfg.spectrogram.n_fft,
    hop_length=binary_cfg.spectrogram.hop_length,
    n_mels=binary_cfg.spectrogram.n_mels,
    top_db=binary_cfg.spectrogram.top_db,
    spectrograms_path=str(SPEC_DIR),
    save_npy=True,
    fill_highfreq=binary_cfg.spectrogram.fill_highfreq,
    noise_db_std=binary_cfg.spectrogram.noise_db_std,
    storage_dtype=binary_cfg.spectrogram.storage_dtype,
)
print(f"Spectrograms saved to: {SPEC_DIR}")

Checking for existing spectrograms...


Checking files: 100%|██████████| 5/5 [00:00<00:00, 902.97it/s]

Found 1006/1006 existing spectrograms
Need to create 0 spectrograms from 0 audio files
All spectrograms already exist! Skipping computation.
Spectrograms saved to: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/spectrograms


In [14]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold


def create_splits(windows, spectrograms_dir, sounds_list, output_dir, cfg):
    """Build train/val/test CSVs with spec_name = {sound_filename}_{start}_{end}.npy."""
    sounds_stems = {
        s["id"]: os.path.splitext(os.path.basename(s["file_name_path"]))[0]
        for s in sounds_list
    }
    df = pd.DataFrame(windows)
    df["sound_filename"] = df["sound_id"].map(sounds_stems)
    df["spec_name"] = df.apply(
        lambda r: f"{r['sound_filename']}_{r['start']}_{r['end']}.npy", axis=1
    )
    df["spec_exists"] = df["spec_name"].apply(
        lambda x: os.path.exists(os.path.join(spectrograms_dir, x))
    )
    df = df[df["spec_exists"]].drop(columns=["spec_exists"])
    print(f"Windows with existing spectrograms: {len(df)}")

    gss = GroupShuffleSplit(
        n_splits=1, test_size=cfg.splits.test_size,
        random_state=cfg.splits.random_state
    )
    trainval_idx, test_idx = next(gss.split(df, df["label"], groups=df["sound_id"]))
    trainval_df = df.iloc[trainval_idx].copy()
    test_df     = df.iloc[test_idx].copy()

    sgkf = StratifiedGroupKFold(
        n_splits=cfg.splits.n_splits, shuffle=True,
        random_state=cfg.splits.random_state
    )
    train_idx, val_idx = next(
        sgkf.split(trainval_df, trainval_df["label"], trainval_df["sound_id"])
    )
    train_df = trainval_df.iloc[train_idx].copy()
    val_df   = trainval_df.iloc[val_idx].copy()

    os.makedirs(output_dir, exist_ok=True)
    train_df.to_csv(os.path.join(output_dir, "train_split.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir,   "val_split.csv"),   index=False)
    test_df.to_csv(os.path.join(output_dir,  "test_split.csv"),  index=False)

    print(f"\nSplit sizes  (train={len(train_df)}  val={len(val_df)}  test={len(test_df)})")
    for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(f"  {name}: label dist = {split['label'].value_counts().to_dict()}")
    return train_df, val_df, test_df


binary_train, binary_val, binary_test = create_splits(
    binary_windows, str(SPEC_DIR), anno_data["sounds"],
    str(BINARY_DIR), binary_cfg
)

Windows with existing spectrograms: 1006

Split sizes  (train=601  val=192  test=213)
  train: label dist = {1: 303, 0: 298}
  val: label dist = {0: 112, 1: 80}
  test: label dist = {1: 120, 0: 93}


In [15]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

from PytorchWildlife.models.bioacoustics import ResNetClassifier
from train import SpectrogramDataModule, DataModuleConfig

pl.seed_everything(42)

# ── DataModule ───────────────────────────────────────────────────────────────
binary_dm_cfg = DataModuleConfig(
    train_csv=str(BINARY_DIR / "train_split.csv"),
    val_csv  =str(BINARY_DIR / "val_split.csv"),
    test_csv =str(BINARY_DIR / "test_split.csv"),
    root     =str(SPEC_DIR),
    target_size=binary_cfg.training.target_size,
    batch_size =binary_cfg.training.batch_size,
    num_workers=binary_cfg.training.num_workers,
    use_specaug=binary_cfg.training.use_specaug,
    normalize  =binary_cfg.training.normalize,
    num_classes=None,    # None → auto-detect (binary)
    use_mixup  =True,
    pin_memory =False,   # safer on CPU
)
binary_dm = SpectrogramDataModule(binary_dm_cfg)
binary_dm.setup()

# ── Model ────────────────────────────────────────────────────────────────────
binary_model = ResNetClassifier(
    num_classes     =binary_dm.num_classes,
    in_channels     =binary_dm.in_channels,
    backbone        =binary_cfg.training.backbone,
    lr              =binary_cfg.training.lr,
    weight_decay    =binary_cfg.training.weight_decay,
    T_max           =binary_cfg.training.epochs,
    batch_size      =binary_cfg.training.batch_size,
    pos_weight      =binary_cfg.training.pos_weight,
    conf_threshold  =binary_cfg.training.conf_threshold,
    class_names     =list(binary_cfg.class_names.values()),
)
print(f"Mode     : {'Binary' if binary_dm.is_binary else 'Multiclass'}")
print(f"Classes  : {binary_dm.num_classes}")
print(f"Channels : {binary_dm.in_channels}")

# ── Trainer ──────────────────────────────────────────────────────────────────
binary_logger = CSVLogger(str(BINARY_DIR), name="logs")
binary_ckpt   = ModelCheckpoint(
    monitor="val/f1", mode="max", save_top_k=1,
    dirpath=str(BINARY_DIR / "checkpoints"),
    filename="binary-{epoch:02d}-{val/f1:.3f}",
)
binary_trainer = pl.Trainer(
    max_epochs        =binary_cfg.training.epochs,
    accelerator       ="auto",   # GPU if available, else CPU
    devices           =1,
    precision         ="32",     # 16-mixed requires GPU
    gradient_clip_val =1.0,
    log_every_n_steps =5,
    callbacks         =[binary_ckpt, LearningRateMonitor()],
    logger            =binary_logger,
    enable_progress_bar=True,
)

binary_trainer.fit(binary_model, datamodule=binary_dm)
binary_trainer.test(binary_model, datamodule=binary_dm, ckpt_path="best")
print(f"\nBest checkpoint : {binary_ckpt.best_model_path}")
print(f"Best val/f1     : {binary_ckpt.best_model_score:.4f}")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA H100 NVL MIG 3g.47gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Mode     : Binary
Classes  : 2
Channels : 1


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name         ┃ Type                       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ net          │ ResNet                     │ 11.2 M │ train │     0 │
│ 1  │ criterion    │ BCEWithLogitsLoss          │      0 │ train │     0 │
│ 2  │ train_acc    │ BinaryAccuracy             │      0 │ train │     0 │
│ 3  │ val_acc      │ BinaryAccuracy             │      0 │ train │     0 │
│ 4  │ test_acc     │ BinaryAccuracy             │      0 │ train │     0 │
│ 5  │ train_f1     │ BinaryF1Score              │      0 │ train │     0 │
│ 6  │ val_f1       │ BinaryF1Score              │      0 │ train │     0 │
│ 7  │ test_f1      │ BinaryF1Score              │      0 │ train │     0 │
│ 8  │ train_auprc  │ BinaryAveragePrecision     │      0 │ train │     0 │
│ 9  │ val_auprc    │ BinaryAveragePrecision     │      0 │ train │     0 │
│ 10 │ test_auprc   │ BinaryAveragePrecision     │      0 │ train │     0 │
│ 11 │ test_prec    │ BinaryPrecision            │      0 │ train │     0 │
│ 12 │ test_rec     │ BinaryRecall               │      0 │ train │     0 │
│ 13 │ test_cm      │ BinaryConfusionMatrix      │      0 │ train │     0 │
│ 14 │ test_prcurve │ BinaryPrecisionRecallCurve │      0 │ train │     0 │
└────┴──────────────┴────────────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 82                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=10` reached.


Restoring states from the checkpoint path at /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/binary/checkpoints/binary-epoch=00-val/f1=0.957-v1.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/binary/checkpoints/binary-epoch=00-val/f1=0.957-v1.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/acc_neg        │    0.9354838728904724     │
│       test/acc_pos        │    0.9416666626930237     │
│        test/auprc         │    0.9940669536590576     │
│          test/f1          │    0.9456067085266113     │
│         test/loss         │    0.1543571650981903     │
│      test/precision       │    0.9495798349380493     │
│        test/recall        │    0.9416666626930237     │
└───────────────────────────┴───────────────────────────┘


Best checkpoint : /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/binary/checkpoints/binary-epoch=00-val/f1=0.957-v1.ckpt
Best val/f1     : 0.9565


In [17]:
# Plot training curves from CSVLogger output
metrics_path = Path(binary_logger.log_dir) / "metrics.csv"
metrics = pd.read_csv(metrics_path)

train_metrics = metrics[metrics["train/loss"].notna()]
val_metrics   = metrics[metrics["val/loss"].notna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(train_metrics["epoch"], train_metrics["train/loss"], label="train")
ax1.plot(val_metrics["epoch"],   val_metrics["val/loss"],     label="val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Binary — Loss")
ax1.legend()

ax2.plot(val_metrics["epoch"], val_metrics["val/f1"], color="#1976D2")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1")
ax2.set_title("Binary — Validation F1")

plt.tight_layout()
plt.show()

## 4. Multiclass Classification

**Task:** classify windows into one of 5 categories — `noise`, `CACCEL`, `CYAVIO`, `PSAANG`, `RAMCAR`.

Key differences from binary:
- `build_windows(multiclass=True)` → `label = category_id` (0–4) instead of 0/1
- `ResNetClassifier(num_classes=5)` → CrossEntropyLoss + softmax instead of BCEWithLogitsLoss
- Spectrograms are **shared** — only new windows (if any) are computed

In [18]:
MULTICLASS_CONFIG_PATH = str(DEMO_DIR / "config" / "ppa4_multiclass.yaml")

multiclass_cfg_dict = {
    "name": "ppa4_multiclass",
    "datasets": ["audios"],
    "class_names": {0: "noise", 1: "CACCEL", 2: "CYAVIO", 3: "PSAANG", 4: "RAMCAR"},
    "paths": {
        "data_root":        str(DATA_DIR),
        "output_root":      str(MULTICLASS_DIR),
        "spectrograms_dir": str(SPEC_DIR),   # reuse binary spectrograms
        "annotations_file": "multiclass_annotations.json",
    },
    "audio": {
        "sample_rate":         48000,
        "window_size_sec":     5.0,
        "overlap_sec":         4.0,
        "window_strategy":     "balanced",
        "negative_proportion": 0.5,
        "multiclass":          True,   # label = category_id
    },
    "spectrogram": {
        "n_fft":         2048,
        "hop_length":    512,
        "n_mels":        128,
        "top_db":        80.0,
        "fill_highfreq": False,
        "noise_db_std":  3.0,
        "storage_dtype": "float32",
    },
    "training": {
        "batch_size":     16,
        "num_workers":    2,
        "lr":             1e-4,
        "weight_decay":   1e-4,
        "epochs":         10,
        "backbone":       "resnet18",
        "num_classes":    5,
        "target_size":    [128, 465],
        "normalize":      True,
        "use_specaug":    True,
    },
    "splits": {
        "test_size":    0.2,
        "val_size":     0.2,
        "n_splits":     3,
        "random_state": 42,
    },
}

with open(MULTICLASS_CONFIG_PATH, "w") as f:
    yaml.dump(multiclass_cfg_dict, f, default_flow_style=False, sort_keys=False)

multiclass_cfg = load_config(MULTICLASS_CONFIG_PATH)
print(f"Config loaded: {multiclass_cfg.name}")
print(f"  num_classes : {multiclass_cfg.training.num_classes}")
print(f"  multiclass  : {multiclass_cfg.audio.multiclass}")

Config loaded: ppa4_multiclass
  num_classes : 5
  multiclass  : True


In [19]:
multiclass_windows = run_windows(multiclass_cfg)
print(f"\nLabel distribution: {pd.Series([w['label'] for w in multiclass_windows]).value_counts().to_dict()}")


Step: Build Windows
Loading existing windows from: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/multiclass/windows_mapping_4.0overlap.json
Loaded 332 windows

Label distribution: {1: 166, 0: 166}

Label distribution: {1: 166, 0: 166}


In [20]:
# Load multiclass annotations to get sound paths
with open(multiclass_cfg.paths.annotations_path) as f:
    mc_anno_data = json.load(f)
mc_sounds_map = {s["id"]: s for s in mc_anno_data["sounds"]}

mc_inf_windows = [
    {
        "window_id":  w["window_id"],
        "sound_path": mc_sounds_map[w["sound_id"]]["file_name_path"],
        "start":      w["start"],
        "end":        w["end"],
    }
    for w in multiclass_windows
    if w["sound_id"] in mc_sounds_map
]

# Existing spectrograms are skipped automatically
compute_mel_spectrograms_gpu(
    windows=mc_inf_windows,
    sample_rate=multiclass_cfg.audio.sample_rate,
    n_fft=multiclass_cfg.spectrogram.n_fft,
    hop_length=multiclass_cfg.spectrogram.hop_length,
    n_mels=multiclass_cfg.spectrogram.n_mels,
    top_db=multiclass_cfg.spectrogram.top_db,
    spectrograms_path=str(SPEC_DIR),
    save_npy=True,
    fill_highfreq=multiclass_cfg.spectrogram.fill_highfreq,
    noise_db_std=multiclass_cfg.spectrogram.noise_db_std,
    storage_dtype=multiclass_cfg.spectrogram.storage_dtype,
)

Checking for existing spectrograms...


Checking files: 100%|██████████| 5/5 [00:00<00:00, 2423.89it/s]

Found 332/332 existing spectrograms
Need to create 0 spectrograms from 0 audio files
All spectrograms already exist! Skipping computation.


In [21]:
multiclass_train, multiclass_val, multiclass_test = create_splits(
    multiclass_windows, str(SPEC_DIR), mc_anno_data["sounds"],
    str(MULTICLASS_DIR), multiclass_cfg
)

Windows with existing spectrograms: 332

Split sizes  (train=190  val=61  test=81)
  train: label dist = {0: 100, 1: 90}
  val: label dist = {0: 32, 1: 29}
  test: label dist = {1: 47, 0: 34}


In [22]:
pl.seed_everything(42)

mc_dm_cfg = DataModuleConfig(
    train_csv  =str(MULTICLASS_DIR / "train_split.csv"),
    val_csv    =str(MULTICLASS_DIR / "val_split.csv"),
    test_csv   =str(MULTICLASS_DIR / "test_split.csv"),
    root       =str(SPEC_DIR),
    target_size=multiclass_cfg.training.target_size,
    batch_size =multiclass_cfg.training.batch_size,
    num_workers=multiclass_cfg.training.num_workers,
    use_specaug=multiclass_cfg.training.use_specaug,
    normalize  =multiclass_cfg.training.normalize,
    num_classes=multiclass_cfg.training.num_classes,  # 5 — disables MixUp
    use_mixup  =False,
    pin_memory =False,
)
mc_dm = SpectrogramDataModule(mc_dm_cfg)
mc_dm.setup()

mc_model = ResNetClassifier(
    num_classes  =multiclass_cfg.training.num_classes,
    in_channels  =mc_dm.in_channels,
    backbone     =multiclass_cfg.training.backbone,
    lr           =multiclass_cfg.training.lr,
    weight_decay =multiclass_cfg.training.weight_decay,
    T_max        =multiclass_cfg.training.epochs,
    batch_size   =multiclass_cfg.training.batch_size,
    class_names  =list(multiclass_cfg.class_names.values()),
)
print(f"Mode    : {'Binary' if mc_dm.is_binary else 'Multiclass'}")
print(f"Classes : {mc_dm.num_classes}")

mc_logger = CSVLogger(str(MULTICLASS_DIR), name="logs")
mc_ckpt   = ModelCheckpoint(
    monitor="val/f1", mode="max", save_top_k=1,
    dirpath=str(MULTICLASS_DIR / "checkpoints"),
    filename="multiclass-{epoch:02d}-{val/f1:.3f}",
)
mc_trainer = pl.Trainer(
    max_epochs        =multiclass_cfg.training.epochs,
    accelerator       ="auto",
    devices           =1,
    precision         ="32",
    gradient_clip_val =1.0,
    log_every_n_steps =5,
    callbacks         =[mc_ckpt, LearningRateMonitor()],
    logger            =mc_logger,
    enable_progress_bar=True,
)

mc_trainer.fit(mc_model, datamodule=mc_dm)
mc_trainer.test(mc_model, datamodule=mc_dm, ckpt_path="best")
print(f"\nBest checkpoint : {mc_ckpt.best_model_path}")
print(f"Best val/f1     : {mc_ckpt.best_model_score:.4f}")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Mode    : Multiclass
Classes : 5


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ net                 │ ResNet                    │ 11.2 M │ train │     0 │
│ 1  │ criterion           │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2  │ train_acc           │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3  │ val_acc             │ MulticlassAccuracy        │      0 │ train │     0 │
│ 4  │ test_acc            │ MulticlassAccuracy        │      0 │ train │     0 │
│ 5  │ train_f1            │ MulticlassF1Score         │      0 │ train │     0 │
│ 6  │ val_f1              │ MulticlassF1Score         │      0 │ train │     0 │
│ 7  │ test_f1             │ MulticlassF1Score         │      0 │ train │     0 │
│ 8  │ test_prec           │ MulticlassPrecision       │      0 │ train │     0 │
│ 9  │ test_rec            │ MulticlassRecall          │      0 │ train │     0 │
│ 10 │ test_cm             │ MulticlassConfusionMatrix │      0 │ train │     0 │
│ 11 │ test_f1_per_class   │ MulticlassF1Score         │      0 │ train │     0 │
│ 12 │ test_prec_per_class │ MulticlassPrecision       │      0 │ train │     0 │
│ 13 │ test_rec_per_class  │ MulticlassRecall          │      0 │ train │     0 │
└────┴─────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 81                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=10` reached.


Restoring states from the checkpoint path at /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/multiclass/checkpoints/multiclass-epoch=05-val/f1=0.902-v1.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/multiclass/checkpoints/multiclass-epoch=05-val/f1=0.902-v1.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃         Test metric          ┃         DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test/acc_micro        │      0.9629629850387573      │
│           test/f1            │      0.9618224501609802      │
│          test/loss           │     0.15738801658153534      │
│ test/macro_average_precision │             nan              │
│        test/precision        │      0.9640151262283325      │
│         test/recall          │      0.9599499702453613      │
└──────────────────────────────┴──────────────────────────────┘


Best checkpoint : /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/multiclass/checkpoints/multiclass-epoch=05-val/f1=0.902-v1.ckpt
Best val/f1     : 0.9016


In [24]:
mc_metrics = pd.read_csv(Path(mc_logger.log_dir) / "metrics.csv")
mc_train   = mc_metrics[mc_metrics["train/loss"].notna()]
mc_val     = mc_metrics[mc_metrics["val/loss"].notna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(mc_train["epoch"], mc_train["train/loss"], label="train")
ax1.plot(mc_val["epoch"],   mc_val["val/loss"],     label="val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Multiclass — Loss")
ax1.legend()

ax2.plot(mc_val["epoch"], mc_val["val/f1"], color="#388E3C")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1 (macro)")
ax2.set_title("Multiclass — Validation F1")

plt.tight_layout()
plt.show()

## 5. Inference

Run both trained models on a held-out audio file to produce sliding-window predictions,
then visualise the probability timeline against the ground-truth annotations.

The inference pipeline:
1. `build_inference_windows()` — sliding windows over the whole file
2. `compute_mel_spectrograms_gpu()` — saves to a separate `inference/spectrograms/` directory
3. `run_inference_batch()` — returns probabilities per window
4. `save_inference_results()` — writes predictions CSV

In [25]:
from inference import (
    build_inference_windows,
    load_model_from_checkpoint,
    run_inference_batch,
    save_inference_results,
    BioacousticsInferenceDataset,
)
from torch.utils.data import DataLoader

# Use G040_timelapse_20250629 as the inference target
INFER_FILE = str(AUDIOS_DIR / "G040_timelapse_20250629.wav")
INFER_DIR  = OUTPUT_DIR / "inference"
INFER_SPEC = INFER_DIR / "spectrograms"
INFER_DIR.mkdir(parents=True, exist_ok=True)
INFER_SPEC.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 48000
WIN_SEC = 5.0
OVL_SEC = 4.0

print(f"Inference target : {os.path.basename(INFER_FILE)}")
print(f"Device           : {DEVICE}")

# Build sliding windows (covers entire file)
infer_windows = build_inference_windows(
    audios_source=INFER_FILE,
    window_size_sec=WIN_SEC,
    overlap_sec=OVL_SEC,
    sample_rate=SR,
)
print(f"Inference windows: {len(infer_windows)}")

# Compute spectrograms
compute_mel_spectrograms_gpu(
    windows=infer_windows,
    sample_rate=SR,
    n_fft=2048, hop_length=512, n_mels=128, top_db=80.0,
    spectrograms_path=str(INFER_SPEC),
    save_npy=True, fill_highfreq=False,
)

# Build spec_name column (inference naming: {sound_filename}_{start}_{end}.npy)
sound_stem = os.path.splitext(os.path.basename(INFER_FILE))[0]
infer_df = pd.DataFrame(infer_windows)
infer_df["spec_name"] = infer_df.apply(
    lambda r: str(INFER_SPEC / f"{sound_stem}_{r['start']}_{r['end']}.npy"), axis=1
)
print(f"Spectrogram files ready")

Inference target : G040_timelapse_20250629.wav
Device           : cuda
Inference windows: 476
Checking for existing spectrograms...


Checking files: 100%|██████████| 1/1 [00:00<00:00, 294.28it/s]


Found 0/476 existing spectrograms
Need to create 476 spectrograms from 1 audio files


Processing files: 100%|██████████| 1/1 [00:02<00:00,  2.02s/it]


Spectrogram files ready


In [26]:
TARGET_SIZE = binary_cfg.training.target_size

infer_ds = BioacousticsInferenceDataset(
    dataframe=infer_df, x_col="spec_name",
    target_size=TARGET_SIZE, normalize=True,
)
infer_dl = DataLoader(infer_ds, batch_size=32, shuffle=False, num_workers=0)

# ── Binary inference ─────────────────────────────────────────────────────────
binary_infer_model = load_model_from_checkpoint(binary_ckpt.best_model_path, device=DEVICE)

binary_results = run_inference_batch(
    model=binary_infer_model,
    dataloader=infer_dl,
    sample_rate=SR,
    num_classes=2,
    device=DEVICE,
)

binary_pred_path = str(INFER_DIR / "binary_predictions.csv")
binary_pred_df = save_inference_results(
    results=binary_results,
    output_path=binary_pred_path,
    num_classes=2,
    class_names=["noise", "AVEVOC"],
)
binary_pred_df.head()

Loading model from checkpoint: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/binary/checkpoints/binary-epoch=00-val/f1=0.957-v1.ckpt
Running inference on 15 batches...
Mode: binary


100%|██████████| 15/15 [00:00<00:00, 38.61it/s]

Results saved to: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/inference/binary_predictions.csv


,audio,start(s),end(s),prediction,probability,confidence
402,G040_timelapse_20250629,402.0,407.0,0,0.001201,0.997599
400,G040_timelapse_20250629,400.0,405.0,0,0.001388,0.997225
403,G040_timelapse_20250629,403.0,408.0,0,0.001414,0.997171
404,G040_timelapse_20250629,404.0,409.0,0,0.001458,0.997084
463,G040_timelapse_20250629,463.0,468.0,0,0.001494,0.997013


In [27]:
# ── Multiclass inference ──────────────────────────────────────────────────────
mc_infer_model = load_model_from_checkpoint(mc_ckpt.best_model_path, device=DEVICE)

mc_results = run_inference_batch(
    model=mc_infer_model,
    dataloader=infer_dl,
    sample_rate=SR,
    num_classes=5,
    device=DEVICE,
)

mc_pred_path = str(INFER_DIR / "multiclass_predictions.csv")
mc_pred_df = save_inference_results(
    results=mc_results,
    output_path=mc_pred_path,
    num_classes=5,
    class_names=["noise", "CACCEL", "CYAVIO", "PSAANG", "RAMCAR"],
)
mc_pred_df.head()

Loading model from checkpoint: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/multiclass/checkpoints/multiclass-epoch=05-val/f1=0.902-v1.ckpt
Running inference on 15 batches...
Mode: multiclass (5 classes)


100%|██████████| 15/15 [00:00<00:00, 62.04it/s]

Results saved to: /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/inference/multiclass_predictions.csv


,file_path,audio,start(s),end(s),prediction,noise_prob,CACCEL_prob,CYAVIO_prob,PSAANG_prob,RAMCAR_prob
0,/home/v-druizlopez/CameraTraps/PW_Bioacoustics...,G040_timelapse_20250629,0.0,5.0,0,0.996287,0.000197,0.000828,0.001704,0.000984
1,/home/v-druizlopez/CameraTraps/PW_Bioacoustics...,G040_timelapse_20250629,1.0,6.0,0,0.997312,0.000117,0.000595,0.001225,0.000752
2,/home/v-druizlopez/CameraTraps/PW_Bioacoustics...,G040_timelapse_20250629,2.0,7.0,0,0.995474,0.000469,0.000933,0.001865,0.001259
3,/home/v-druizlopez/CameraTraps/PW_Bioacoustics...,G040_timelapse_20250629,3.0,8.0,0,0.623293,0.213181,0.023453,0.095798,0.044275
4,/home/v-druizlopez/CameraTraps/PW_Bioacoustics...,G040_timelapse_20250629,4.0,9.0,0,0.839655,0.082686,0.016051,0.041049,0.020558


In [28]:
# Load ground-truth annotations for the inference file
gt_df = pd.read_csv(
    LABELS_DIR / "G040_timelapse_20250629.Table.1.selections.txt",
    delimiter="\t"
)

# Restrict to first 120 s for a readable plot
T_MAX = 120.0
binary_plot  = binary_pred_df[binary_pred_df["end(s)"] <= T_MAX].copy()
mc_plot      = mc_pred_df[mc_pred_df["end(s)"] <= T_MAX].copy()
gt_plot      = gt_df[gt_df["End Time (s)"] <= T_MAX].copy()

SPECIES_COLORS = {
    "CACCEL": "#E53935",
    "CYAVIO": "#8E24AA",
    "PSAANG": "#FB8C00",
    "RAMCAR": "#00897B",
}
CLASS_NAMES = ["noise", "CACCEL", "CYAVIO", "PSAANG", "RAMCAR"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ── Binary panel ─────────────────────────────────────────────────────────────
t_mid = (binary_plot["start(s)"] + binary_plot["end(s)"]) / 2
ax1.plot(t_mid, binary_plot["probability"], color="#1976D2", lw=0.8)
ax1.axhline(0.5, color="grey", lw=0.8, ls="--")
ax1.set_ylim(-0.05, 1.05)
ax1.set_ylabel("P(AVEVOC)")
ax1.set_title("Binary inference — G040_timelapse_20250629.wav (first 120 s)")

for _, row in gt_plot.iterrows():
    ax1.axvspan(row["Begin Time (s)"], row["End Time (s)"],
                alpha=0.15, color="#1976D2")

# ── Multiclass panel ─────────────────────────────────────────────────────────
t_mid_mc = (mc_plot["start(s)"] + mc_plot["end(s)"]) / 2
for i, cls in enumerate(CLASS_NAMES[1:], start=1):  # skip noise
    col = cls.lower().replace("/", "_") + "_prob"
    if col in mc_plot.columns:
        ax2.plot(t_mid_mc, mc_plot[col],
                 color=SPECIES_COLORS.get(cls, "grey"), lw=0.8, label=cls)

# Shade ground-truth species annotations
for _, row in gt_plot.iterrows():
    det = str(row.get("Determination", "")).strip()
    if det in SPECIES_COLORS:
        ax2.axvspan(row["Begin Time (s)"], row["End Time (s)"],
                    alpha=0.15, color=SPECIES_COLORS[det])

ax2.set_ylim(-0.05, 1.05)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Class probability")
ax2.set_title("Multiclass inference")
ax2.legend(loc="upper right", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

print("Inference results saved to:")
print(f"  {binary_pred_path}")
print(f"  {mc_pred_path}")

Inference results saved to:
  /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/inference/binary_predictions.csv
  /home/v-druizlopez/CameraTraps/PW_Bioacoustics/demo/output/inference/multiclass_predictions.csv
